# Phase 3 — Renewable Share Forecast to 2030

This notebook forecasts monthly renewable generation share using Holt-Winters exponential smoothing and validates the model on 2023-2024 holdout data.

In [1]:

import subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from statsmodels.tsa.holtwinters import ExponentialSmoothing
PROJECT_ROOT=Path.cwd()
if PROJECT_ROOT.name=='notebooks': PROJECT_ROOT=PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0,str(PROJECT_ROOT))
CHROME_PATH=PROJECT_ROOT/'.kaleido_chrome/chrome-mac-arm64/Google Chrome for Testing.app/Contents/MacOS/Google Chrome for Testing'
CHART_DIR=PROJECT_ROOT/'outputs/charts'; CHART_DIR.mkdir(parents=True, exist_ok=True)
def save_chart(fig,filename):
    html_path=CHART_DIR/f'{filename}.html'; png_path=CHART_DIR/f'{filename}.png'
    fig.write_html(html_path); print(f'Saved: {html_path.relative_to(PROJECT_ROOT)}')
    subprocess.run([str(CHROME_PATH),'--headless=new','--disable-gpu','--no-sandbox','--disable-dev-shm-usage',f'--screenshot={png_path}','--window-size=1200,700',f'file://{html_path}'],check=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE,text=True)
    print(f'Saved: {png_path.relative_to(PROJECT_ROOT)}')
renewable_df=pd.read_csv(PROJECT_ROOT/'data/processed/renewable_share.csv', parse_dates=['month_date'])
print(f'Loaded renewable share dataset: {renewable_df.shape}')
print(renewable_df.head(10).to_string(index=False))
print('Chart export method: Plotly HTML + approved headless Chrome PNG screenshot')


Loaded renewable share dataset: (120, 14)
 year  month month_date  total_generation_mwh  renewable_mwh  wind_mwh  solar_mwh    gas_mwh   coal_mwh  renewable_pct  solar_pct  wind_pct  renewable_pct_prior_year  renewable_yoy_change
 2015      1 2015-01-01            38164000.0      3063000.0 3031000.0    32000.0 19252000.0 11365000.0          8.026      0.084     7.942                       NaN                   NaN
 2015      2 2015-02-01            33449000.0      3301000.0 3268000.0    33000.0 16980000.0  9146000.0          9.869      0.099     9.770                       NaN                   NaN
 2015      3 2015-03-01            33042000.0      2586000.0 2544000.0    42000.0 18801000.0  7403000.0          7.826      0.127     7.699                       NaN                   NaN
 2015      4 2015-04-01            32009000.0      4144000.0 4099000.0    45000.0 17317000.0  7224000.0         12.946      0.141    12.806                       NaN                   NaN
 2015      5 2015-

In [2]:

print('RENEWABLE SHARE FORECAST TO 2030')
print('='*50)
renewable_series=renewable_df.set_index('month_date')['renewable_pct'].sort_index()
renewable_series.index=pd.DatetimeIndex(renewable_series.index, freq='MS')
train=renewable_series[:'2022-12']; test=renewable_series['2023-01':]
model=ExponentialSmoothing(train, trend='add', seasonal='add', seasonal_periods=12).fit(optimized=True)
test_pred=model.forecast(len(test))
rmse=np.sqrt(((test.values-test_pred.values)**2).mean())
print(f'Holdout RMSE (2023-2024): {rmse:.2f} percentage points')
print(f'AIC: {model.aic:.1f}')
print(f'BIC: {model.bic:.1f}')
model_full=ExponentialSmoothing(renewable_series, trend='add', seasonal='add', seasonal_periods=12).fit(optimized=True)
forecast_values=model_full.forecast(72)
forecast_df=pd.DataFrame({'date': forecast_values.index, 'forecast_pct': forecast_values.values, 'lower_ci': forecast_values.values-1.96*rmse, 'upper_ci': forecast_values.values+1.96*rmse})
forecast_df['lower_ci']=forecast_df['lower_ci'].clip(lower=0); forecast_df['upper_ci']=forecast_df['upper_ci'].clip(upper=100)
print('\nForecast table sample:')
print(forecast_df.head(12).to_string(index=False))
print(f"\nForecast results:")
print(f"  Renewable share Dec 2027: {forecast_df[forecast_df['date'].dt.year==2027]['forecast_pct'].iloc[-1]:.1f}%")
dec_2030=forecast_df[forecast_df['date'].dt.year==2030]['forecast_pct'].iloc[-1]
print(f"  Renewable share Dec 2030: {dec_2030:.1f}%")
cross_50=forecast_df[forecast_df['forecast_pct']>=50]
if len(cross_50)>0:
    print(f"  Projected to cross 50%: {cross_50.iloc[0]['date'].strftime('%B %Y')}")
else:
    print(f"  Does not reach 50% by 2030 (max: {forecast_df['forecast_pct'].max():.1f}%)")
forecast_df.to_csv(PROJECT_ROOT/'outputs/tableau_exports/renewable_forecast.csv', index=False)
print('Saved forecast CSV: outputs/tableau_exports/renewable_forecast.csv')


RENEWABLE SHARE FORECAST TO 2030
Holdout RMSE (2023-2024): 3.05 percentage points
AIC: 187.8
BIC: 228.8

Forecast table sample:
      date  forecast_pct  lower_ci  upper_ci
2025-01-01     31.776896 25.807939 37.745854
2025-02-01     34.308087 28.339130 40.277045
2025-03-01     37.214725 31.245767 43.183682
2025-04-01     38.283920 32.314962 44.252877
2025-05-01     34.233435 28.264478 40.202392
2025-06-01     30.092419 24.123461 36.061376
2025-07-01     27.704180 21.735223 33.673138
2025-08-01     26.340479 20.371521 32.309436
2025-09-01     27.383976 21.415019 33.352933
2025-10-01     33.052446 27.083488 39.021403
2025-11-01     34.582545 28.613587 40.551502
2025-12-01     33.901189 27.932231 39.870146

Forecast results:
  Renewable share Dec 2027: 38.4%
  Renewable share Dec 2030: 45.1%
  Does not reach 50% by 2030 (max: 49.4%)
Saved forecast CSV: outputs/tableau_exports/renewable_forecast.csv


In [3]:

fig=go.Figure()
fig.add_trace(go.Scatter(x=renewable_series.index, y=renewable_series.values, name='Historical', line=dict(color='#4CAF50',width=2)))
fig.add_trace(go.Scatter(x=forecast_df['date'], y=forecast_df['forecast_pct'], name='Forecast', line=dict(color='#4CAF50',width=2,dash='dash')))
fig.add_trace(go.Scatter(x=pd.concat([forecast_df['date'], forecast_df['date'][::-1]]), y=pd.concat([forecast_df['upper_ci'], forecast_df['lower_ci'][::-1]]), fill='toself', fillcolor='rgba(76,175,80,0.15)', line=dict(width=0), name='95% Confidence Interval'))
for level in [40,50,60]:
    fig.add_hline(y=level, line_dash='dot', line_color='lightgray', annotation_text=f'{level}%', annotation_position='right')
start=pd.Timestamp('2025-01-01')
fig.add_shape(type='line', x0=start, x1=start, y0=0, y1=1, xref='x', yref='paper', line=dict(color='gray', dash='dot'))
fig.add_annotation(x=start, y=1.06, xref='x', yref='paper', text='Forecast begins', showarrow=False, font=dict(size=11, color='gray'))
fig.update_layout(title=dict(text='ERCOT Renewable Share Forecast to 2030<br><sub>Holt-Winters Exponential Smoothing with 95% Confidence Interval</sub>', font=dict(size=18, family='Arial')), font=dict(family='Arial',size=12), xaxis_title='Date', yaxis=dict(title='Renewable Share (%)', ticksuffix='%'), hovermode='x unified', plot_bgcolor='white', paper_bgcolor='white', margin=dict(l=70,r=50,t=90,b=80))
fig.update_xaxes(showgrid=True, gridcolor='#F5F5F5'); fig.update_yaxes(showgrid=True, gridcolor='#F5F5F5')
fig.add_annotation(xref='paper',yref='paper',x=1.0,y=-0.14,text='Source: U.S. EIA; author calculations',showarrow=False,font=dict(size=10,color='gray'),xanchor='right')
save_chart(fig,'chart_5a_renewable_forecast_2030')
fig.show()


Saved: outputs/charts/chart_5a_renewable_forecast_2030.html


Saved: outputs/charts/chart_5a_renewable_forecast_2030.png
